# Benchmark expandido — 100 casos por linguagem

Executa 400 casos no Qwen ou Ministral. Os resultados são gravados no Google Drive após cada resposta, permitindo retomar em outra sessão.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Ative uma GPU T4 no Colab.'

In [ ]:
from google.colab import files, drive
from pathlib import Path
import os
import shutil

uploaded = files.upload()  # selecione Celx-colab-v8.zip
archives = [name for name in uploaded if name.lower().endswith('.zip')]
assert archives, 'Envie Celx-colab-v8.zip.'
repo_dir = Path('/content/legacy-doc-project')
if repo_dir.exists():
    shutil.rmtree(repo_dir)
repo_dir.mkdir(parents=True)
shutil.unpack_archive(f'/content/{archives[0]}', repo_dir)
os.environ['PYTHONPATH'] = str(repo_dir)
drive.mount('/content/drive')
print('Projeto:', repo_dir)

In [ ]:
%cd /content/legacy-doc-project
%pip install -q "transformers>=4.51" "accelerate>=1.0" "bitsandbytes>=0.45" "PyYAML>=6.0"
%pip install -q -e .

In [ ]:
data_phase = Path('/content/drive/MyDrive/Colab Notebooks/Fine-tuning/data_phase')
benchmark_source = data_phase / 'expanded_100.jsonl'
assert benchmark_source.exists(), (
    'Execute primeiro o notebook 02_dados_codexglue_colab.ipynb.'
)
benchmark_local = repo_dir / 'dataset/benchmark/expanded_100.jsonl'
benchmark_local.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(benchmark_source, benchmark_local)

drive_results = data_phase / 'benchmark_100_results'
drive_results.mkdir(parents=True, exist_ok=True)
local_results = repo_dir / 'outputs/benchmark_100'
local_results.parent.mkdir(parents=True, exist_ok=True)
if local_results.exists() or local_results.is_symlink():
    if local_results.is_symlink():
        local_results.unlink()
    else:
        shutil.rmtree(local_results)
local_results.symlink_to(drive_results, target_is_directory=True)
print('Benchmark:', benchmark_local)
print('Resultados persistentes:', drive_results)

## Executar um modelo por sessão

Comece pelo Qwen. Em outra sessão, desmarque Qwen e marque Ministral. Casos já concluídos são ignorados automaticamente.

In [ ]:
import subprocess
import sys

executar_qwen = True  # @param {type:"boolean"}
executar_ministral = False  # @param {type:"boolean"}
selecionados = [
    nome for nome, ativo in {
        'qwen3-1.7b': executar_qwen,
        'ministral3-3b': executar_ministral,
    }.items() if ativo
]
assert len(selecionados) == 1, 'Selecione exatamente um modelo por sessão.'
subprocess.run(
    [sys.executable, 'scripts/run_baseline.py', '--model', selecionados[0]],
    check=True,
    env=os.environ.copy(),
)

In [ ]:
import json
for path in sorted(drive_results.glob('*.jsonl')):
    completed = sum(1 for line in path.open(encoding='utf-8') if line.strip())
    print(f'{path.stem}: {completed}/400 respostas')

In [ ]:
!python scripts/compare_models.py --results outputs/benchmark_100 --output outputs/benchmark_100/model_comparison.csv
import pandas as pd
comparison = drive_results / 'model_comparison.csv'
display(pd.read_csv(comparison))
files.download(str(comparison))